# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import duckdb
import polars as pl
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "data/raw/transjakarta_gps.csv").is_file()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Letakkan transjakarta_gps.csv di data/raw/ dan buka notebook dari proyek.")
DATA_PATH = PROJECT_ROOT / "data/raw/transjakarta_gps.csv"
OUT_DIR = PROJECT_ROOT / "data/processed"
REPORT_DIR = PROJECT_ROOT / "output/milestone2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUT_DIR / "transjakarta_gps_bersih.parquet"
VERSIONS = {"python": platform.python_version(), "polars": pl.__version__, "duckdb": duckdb.__version__}
print(VERSIONS)

{'python': '3.13.0', 'polars': '1.44.2', 'duckdb': '1.5.6'}


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
raw_lf = pl.scan_csv(DATA_PATH, infer_schema=False, row_index_name="source_row")
raw = raw_lf.collect(engine="streaming")
SOURCE_COLS = [c for c in raw.columns if c != "source_row"]
EXPECTED = ["bus_code", "trip_id", "gps_datetime", "location", "dtd", "corridor", "longitude", "latitude", "speed", "course", "color"]
assert SOURCE_COLS == EXPECTED, "Skema input berubah; periksa sebelum melanjutkan."
N_RAW = raw.height
size_mb = DATA_PATH.stat().st_size / 1_000_000
sha = hashlib.sha256()
with DATA_PATH.open("rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(block)
RAW_SHA256 = sha.hexdigest()
SKALA_MEMENUHI = size_mb >= 500 or N_RAW > 1_000_000
print(f"Data: {N_RAW:,} baris, {len(SOURCE_COLS)} kolom; {size_mb:.2f} MB (desimal).")
print("Memenuhi syarat >=500 MB atau >1 juta baris:", SKALA_MEMENUHI)
print("SHA256:", RAW_SHA256)
display(raw.head())

Data: 775,458 baris, 11 kolom; 72.98 MB (desimal).
Memenuhi syarat >=500 MB atau >1 juta baris: False
SHA256: 3429e13bb61429e0429267e76d4070befe39396bcb564781b7e01fef51c9b643


source_row,bus_code,trip_id,gps_datetime,location,dtd,corridor,longitude,latitude,speed,course,color
u32,str,str,str,str,str,str,str,str,str,str,str
0,"""DMR 5049""","""5.112""","""2019-11-26 14:00:29""","""""","""0""","""0""","""106.867523""","""-6.22423""","""0""","""208""","""MERAH KUNING"""
1,"""DMR 5049""","""5.112""","""2019-11-26 14:00:54""","""""","""0""","""0""","""106.867523""","""-6.22423""","""0""","""208""","""MERAH KUNING"""
2,"""DMR 5049""","""5.112""","""2019-11-26 14:01:29""","""""","""0""","""0""","""106.867523""","""-6.22423""","""0""","""208""","""MERAH KUNING"""
3,"""DMR 5049""","""5.112""","""2019-11-26 14:01:59""","""""","""0""","""0""","""106.867523""","""-6.22423""","""0""","""208""","""MERAH KUNING"""
4,"""DMR 5049""","""5.112""","""2019-11-26 14:02:29""","""""","""0""","""0""","""106.867523""","""-6.22423""","""0""","""208""","""MERAH KUNING"""


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
connection = duckdb.connect()
connection.register("dataset_raw", raw.drop("source_row").to_arrow())
profil_raw = connection.execute("SUMMARIZE SELECT * FROM dataset_raw").pl()
profil_raw.write_csv(REPORT_DIR / "profil_raw.csv")
display(profil_raw)

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""bus_code""","""VARCHAR""","""DMR 5049""","""TSW 100""",1764,null,null,null,null,null,775458,0.00
"""trip_id""","""VARCHAR""","""-""","""TR1.3022""",759,null,null,null,null,null,775458,0.00
"""gps_datetime""","""VARCHAR""","""2019-11-26 14:00:00""","""2019-11-26 17:59:59""",14510,null,null,null,null,null,775458,0.00
"""location""","""VARCHAR""","""""","""9-9,9-8""",437,null,null,null,null,null,775458,0.00
"""dtd""","""VARCHAR""","""0""","""1""",160157,null,null,null,null,null,775458,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""longitude""","""VARCHAR""","""106.607""","""108.330973""",107515,null,null,null,null,null,775458,0.00
"""latitude""","""VARCHAR""","""-6.048802""","""-6.791528""",109445,null,null,null,null,null,775458,0.00
"""speed""","""VARCHAR""","""0""","""99.4""",1047,null,null,null,null,null,775458,0.00


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [4]:
MISSING_TOKENS = ["", "-", "null", "none", "nan", "n/a", "na"]
normal = raw.lazy().with_columns([
    pl.when(pl.col(c).str.strip_chars().str.to_lowercase().is_in(MISSING_TOKENS))
      .then(None).otherwise(pl.col(c).str.strip_chars()).alias(c)
    for c in SOURCE_COLS
]).collect(engine="streaming")
NUMERIC = ["dtd", "longitude", "latitude", "speed", "course"]
typed = normal.lazy().with_columns(
    pl.col("gps_datetime").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
    *[pl.col(c).cast(pl.Float64, strict=False) for c in NUMERIC]
).with_columns([
    pl.when(pl.col(c).is_finite()).then(pl.col(c)).otherwise(None).alias(c)
    for c in NUMERIC
]).collect(engine="streaming")
parse_audit = pl.DataFrame([
    {"kolom": col, "gagal_konversi_atau_nonfinite":
     int((normal[col].is_not_null() & typed[col].is_null()).sum())}
    for col in ["gps_datetime", *NUMERIC]
])
def missing_report(frame):
    return pl.DataFrame([
        {"kolom": col, "jumlah_null": frame[col].null_count(),
         "persen_null": round(frame[col].null_count() / frame.height * 100, 3)}
        for col in SOURCE_COLS
    ]).sort("jumlah_null", descending=True)
missing_before = missing_report(typed)
display(parse_audit)
display(missing_before)
connection.register("dataset_typed", typed.drop("source_row").to_arrow())
profil_typed = connection.execute("SUMMARIZE SELECT * FROM dataset_typed").pl()
display(profil_typed)
parse_audit.write_csv(REPORT_DIR / "audit_konversi.csv")
missing_before.write_csv(REPORT_DIR / "missing_sebelum.csv")

kolom,gagal_konversi_atau_nonfinite
str,i64
"""gps_datetime""",0
"""dtd""",0
"""longitude""",0
"""latitude""",0
"""speed""",0
"""course""",0


kolom,jumlah_null,persen_null
str,i64,f64
"""location""",540981,69.763
"""color""",111548,14.385
"""trip_id""",3414,0.44
"""bus_code""",0,0.0
"""gps_datetime""",0,0.0
…,…,…
"""corridor""",0,0.0
"""longitude""",0,0.0
"""latitude""",0,0.0


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""bus_code""","""VARCHAR""","""DMR 5049""","""TSW 100""",1764,null,null,null,null,null,775458,0.00
"""trip_id""","""VARCHAR""","""0""","""TR1.3022""",759,null,null,null,null,null,775458,0.44
"""gps_datetime""","""TIMESTAMP""","""2019-11-26 14:00:00""","""2019-11-26 17:59:59""",15676,"""2019-11-26 16:01:53.919494""",null,"""2019-11-26 14:59:28.831139""","""2019-11-26 16:04:39.692746""","""2019-11-26 17:01:03.237503""",775458,0.00
"""location""","""VARCHAR""","""1-1,1-2""","""9-9,9-8""",437,null,null,null,null,null,775458,69.76
"""dtd""","""DOUBLE""","""0.0""","""1.0""",225059,"""0.16288557937314166""","""0.3037713174815963""","""0.0""","""0.0""","""0.140028712707853""",775458,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""longitude""","""DOUBLE""","""106.607""","""108.330973""",122081,"""106.8347110769981""","""0.062049371105607154""","""106.80057075628115""","""106.83106503331885""","""106.87415742914025""",775458,0.00
"""latitude""","""DOUBLE""","""-6.791528""","""-6.048802""",106981,"""-6.219048020674241""","""0.05685416627670889""","""-6.2520362420008215""","""-6.214362682038829""","""-6.177218794900646""",775458,0.00
"""speed""","""DOUBLE""","""0.0""","""309.0""",1142,"""10.669549479146474""","""14.843973484951007""","""0.0""","""1.9295658914510232""","""19.00513166707375""",775458,0.00


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
MAX_SPEED = 120.0
q1 = typed["speed"].quantile(0.25)
q3 = typed["speed"].quantile(0.75)
iqr = q3 - q1
LOW_IQR, HIGH_IQR = q1 - 1.5 * iqr, q3 + 1.5 * iqr
quality = typed.with_columns(
    pl.col("speed").alias("speed_original"),
    pl.col("course").alias("course_original"),
    ((pl.col("speed") < LOW_IQR) | (pl.col("speed") > HIGH_IQR)).fill_null(False).alias("speed_iqr_outlier"),
    (~pl.col("speed").is_between(0, MAX_SPEED)).fill_null(False).alias("speed_invalid"),
    (~pl.col("course").is_between(0, 360)).fill_null(False).alias("course_invalid"),
    (~(pl.col("longitude").is_between(106, 108) & pl.col("latitude").is_between(-7, -5.5))).fill_null(False).alias("outside_jabodetabek_check")
).with_columns(
    pl.when(pl.col("speed_invalid")).then(None).otherwise(pl.col("speed")).alias("speed"),
    pl.when(pl.col("course_invalid")).then(None)
      .when(pl.col("course") == 360).then(0.0).otherwise(pl.col("course")).alias("course")
)
critical_invalid = (
    pl.col("bus_code").is_null() | pl.col("gps_datetime").is_null()
    | ~pl.col("longitude").is_between(-180, 180).fill_null(False)
    | ~pl.col("latitude").is_between(-90, 90).fill_null(False)
)
quarantine = quality.filter(critical_invalid)
valid = quality.filter(~critical_invalid)
print(f"Q1={q1}; Q3={q3}; IQR={iqr}; batas IQR=[{LOW_IQR}, {HIGH_IQR}]")
display(quality.select(pl.col("speed_iqr_outlier", "speed_invalid", "course_invalid", "outside_jabodetabek_check").sum()))
print(f"Baris dikarantina: {quarantine.height:,}")
quarantine.write_parquet(OUT_DIR / "transjakarta_gps_karantina.parquet")

Q1=0.0; Q3=19.0; IQR=19.0; batas IQR=[-28.5, 47.5]


speed_iqr_outlier,speed_invalid,course_invalid,outside_jabodetabek_check
u32,u32,u32,u32
15337,121,0,8


Baris dikarantina: 0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
KEY = ["bus_code", "gps_datetime"]
duplicate_groups = (
    valid.group_by(KEY).agg(
        pl.len().alias("jumlah_baris"),
        pl.struct([c for c in SOURCE_COLS if c not in KEY]).n_unique().alias("variasi_payload")
    ).filter(pl.col("jumlah_baris") > 1).sort(KEY)
)
duplicate_candidates = valid.join(duplicate_groups.select(KEY), on=KEY, how="inner")
scored = valid.with_columns(
    pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.Int16) for c in SOURCE_COLS]).alias("completeness_score")
)
clean = (
    scored.sort([*KEY, "completeness_score", "source_row"], descending=[False, False, True, False])
    .unique(subset=KEY, keep="first", maintain_order=True)
    .drop("completeness_score").sort(KEY)
)
N_DUP = valid.height - clean.height
N_CONFLICT = duplicate_groups.filter(pl.col("variasi_payload") > 1).height
print(f"Kelompok duplikat: {duplicate_groups.height:,}; kelompok konflik: {N_CONFLICT:,}")
print(f"Baris duplikat dihapus: {N_DUP:,}; baris bersih: {clean.height:,}")
display(duplicate_groups.head(10))
duplicate_groups.write_csv(REPORT_DIR / "kelompok_duplikat.csv")
duplicate_candidates.write_parquet(OUT_DIR / "transjakarta_gps_kandidat_duplikat.parquet")

Kelompok duplikat: 7,750; kelompok konflik: 149
Baris duplikat dihapus: 48,384; baris bersih: 727,074


bus_code,gps_datetime,jumlah_baris,variasi_payload
str,datetime[μs],u32,u32
"""DMR 5049""",2019-11-26 14:16:49,2,1
"""DMR 5049""",2019-11-26 14:19:54,2,1
"""DMR 5049""",2019-11-26 14:24:49,4,1
"""DMR 5049""",2019-11-26 14:48:05,2,1
"""DMR 5049""",2019-11-26 14:48:20,2,1
"""DMR 5049""",2019-11-26 15:05:20,2,1
"""DMR 5049""",2019-11-26 15:22:20,11,1
"""DMR 5049""",2019-11-26 15:35:20,2,1
"""DMR 5049""",2019-11-26 15:41:50,2,1


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [7]:
connection.register("dataset_bersih", clean.to_arrow())
validation = connection.execute(f"""
SELECT COUNT(*) AS total_baris,
 COUNT(*) FILTER (WHERE bus_code IS NULL OR trim(bus_code) = '' OR gps_datetime IS NULL) AS kunci_kosong,
 COUNT(*) FILTER (WHERE longitude IS NULL OR latitude IS NULL
     OR NOT longitude BETWEEN -180 AND 180 OR NOT latitude BETWEEN -90 AND 90) AS koordinat_invalid,
 COUNT(*) FILTER (WHERE speed IS NOT NULL AND NOT speed BETWEEN 0 AND {MAX_SPEED}) AS speed_invalid,
 COUNT(*) FILTER (WHERE course IS NOT NULL AND NOT (course >= 0 AND course < 360)) AS course_invalid
FROM dataset_bersih
""").pl()
duplicate_remaining = connection.execute("""
SELECT COUNT(*) FROM (
 SELECT bus_code, gps_datetime FROM dataset_bersih
 GROUP BY bus_code, gps_datetime HAVING COUNT(*) > 1
)
""").fetchone()[0]
assert clean.height > 0
assert all(validation[c][0] == 0 for c in validation.columns if c != "total_baris")
assert duplicate_remaining == 0
assert N_RAW == clean.height + quarantine.height + N_DUP
assert clean.schema["trip_id"] == pl.String and clean.schema["corridor"] == pl.String
assert clean.schema["gps_datetime"] == pl.Datetime("us")
print("Seluruh validasi LULUS; duplikat kunci tersisa:", duplicate_remaining)
display(validation)
validation.write_csv(REPORT_DIR / "validasi.csv")

Seluruh validasi LULUS; duplikat kunci tersisa: 0


total_baris,kunci_kosong,koordinat_invalid,speed_invalid,course_invalid
i64,i64,i64,i64,i64
727074,0,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
profil_clean = connection.execute("SUMMARIZE SELECT * FROM dataset_bersih").pl()
missing_after = missing_report(clean)
comparison = missing_before.join(missing_after, on="kolom", suffix="_sesudah")
display(comparison)
display(profil_clean)
clean.write_parquet(OUTPUT_PATH, compression="zstd")
# Verifikasi file hasil benar-benar dapat dibaca, termasuk tipe dan nilai.
reloaded = pl.read_parquet(OUTPUT_PATH)
assert reloaded.equals(clean)
assert DATA_PATH.stat().st_size == int(size_mb * 1_000_000)
profil_clean.write_csv(REPORT_DIR / "profil_bersih.csv")
comparison.write_csv(REPORT_DIR / "perbandingan_missing.csv")
metrics = {
    "input": str(DATA_PATH.relative_to(PROJECT_ROOT)), "sha256_raw": RAW_SHA256,
    "ukuran_bytes": DATA_PATH.stat().st_size, "skala_memenuhi": SKALA_MEMENUHI,
    "baris_awal": N_RAW, "baris_karantina": quarantine.height,
    "baris_duplikat_disisihkan": N_DUP, "baris_bersih": clean.height,
    "kelompok_konflik": N_CONFLICT, "speed_invalid_sebelum_dedup": int(quality["speed_invalid"].sum()),
    "speed_iqr_outlier_sebelum_dedup": int(quality["speed_iqr_outlier"].sum()),
    "batas_kecepatan_asumsi": MAX_SPEED, "versi": VERSIONS,
    "periode_awal": str(clean["gps_datetime"].min()), "periode_akhir": str(clean["gps_datetime"].max())
}
(REPORT_DIR / "ringkasan_cleaning.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(json.dumps(metrics, indent=2))
print("Output:", OUTPUT_PATH.relative_to(PROJECT_ROOT))

kolom,jumlah_null,persen_null,jumlah_null_sesudah,persen_null_sesudah
str,i64,f64,i64,f64
"""location""",540981,69.763,500842,68.885
"""color""",111548,14.385,102327,14.074
"""trip_id""",3414,0.44,2800,0.385
"""speed""",0,0.0,99,0.014
"""bus_code""",0,0.0,0,0.0
…,…,…,…,…
"""dtd""",0,0.0,0,0.0
"""corridor""",0,0.0,0,0.0
"""longitude""",0,0.0,0,0.0


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""source_row""","""UINTEGER""","""0""","""775457""",774928,"""381580.27098892274""","""224330.33544485213""","""186175""","""376222""","""574462""",727074,0.00
"""bus_code""","""VARCHAR""","""DMR 5049""","""TSW 100""",1764,null,null,null,null,null,727074,0.00
"""trip_id""","""VARCHAR""","""0""","""TR1.3022""",759,null,null,null,null,null,727074,0.39
"""gps_datetime""","""TIMESTAMP""","""2019-11-26 14:00:00""","""2019-11-26 17:59:59""",15676,"""2019-11-26 16:02:49.817532""",null,"""2019-11-26 15:00:08.63885""","""2019-11-26 16:05:56.893976""","""2019-11-26 17:02:22.263258""",727074,0.00
"""location""","""VARCHAR""","""1-1,1-2""","""9-9,9-8""",437,null,null,null,null,null,727074,68.88
…,…,…,…,…,…,…,…,…,…,…,…
"""course_original""","""DOUBLE""","""0.0""","""360.0""",378,"""185.33397975996942""","""104.54431652683867""","""92.99140254546796""","""183.76962277926415""","""272.2664834092971""",727074,0.00
"""speed_iqr_outlier""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,727074,0.00
"""speed_invalid""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,727074,0.00


{
  "input": "data\\raw\\transjakarta_gps.csv",
  "sha256_raw": "3429e13bb61429e0429267e76d4070befe39396bcb564781b7e01fef51c9b643",
  "ukuran_bytes": 72981601,
  "skala_memenuhi": false,
  "baris_awal": 775458,
  "baris_karantina": 0,
  "baris_duplikat_disisihkan": 48384,
  "baris_bersih": 727074,
  "kelompok_konflik": 149,
  "speed_invalid_sebelum_dedup": 121,
  "speed_iqr_outlier_sebelum_dedup": 15337,
  "batas_kecepatan_asumsi": 120.0,
  "versi": {
    "python": "3.13.0",
    "polars": "1.44.2",
    "duckdb": "1.5.6"
  },
  "periode_awal": "2019-11-26 14:00:00",
  "periode_akhir": "2019-11-26 17:59:59"
}
Output: data\processed\transjakarta_gps_bersih.parquet


## Catatan Milestone 2


Ringkasan berikut dihasilkan dari hasil eksekusi, bukan dari perkiraan di dokumentasi lama. Persentase null sesudah cleaning dapat naik karena nilai invalid diubah menjadi missing dan jumlah baris berubah; itu bukan berarti informasi baru hilang tanpa pencatatan.

In [ ]:
from IPython.display import Markdown
worst = missing_before.row(0, named=True)
summary_text = f"""### Hasil aktual
- Data awal **{N_RAW:,} baris**, ukuran **{size_mb:.2f} MB**, dengan 11 kolom sumber.
- Missing terbesar setelah normalisasi: **{worst['kolom']}**, sebanyak **{worst['jumlah_null']:,}** ({worst['persen_null']:.3f}%).
- Kecepatan di luar asumsi 0–{MAX_SPEED:g}: **{int(quality['speed_invalid'].sum()):,} baris** sebelum deduplikasi; nilai dijadikan null dan nilai asal disimpan.
- Karantina: **{quarantine.height:,} baris**. Duplikat yang disisihkan: **{N_DUP:,} baris**.
- Hasil akhir: **{clean.height:,} baris**; **{N_CONFLICT:,} kelompok** memiliki konflik payload pada kunci yang sama.
- Periode aktual hasil bersih: **{metrics['periode_awal']}** sampai **{metrics['periode_akhir']}**.
- Semua aturan validasi dan uji baca ulang Parquet lulus.

### Keterbatasan dan tindak lanjut
Data yang disertakan **{'memenuhi' if SKALA_MEMENUHI else 'belum memenuhi'}** syarat >=500 MB atau >1 juta baris. Lengkapi data asli sebelum pengumpulan jika belum memenuhi; jangan menduplikasi baris untuk mengejar syarat. Klaim sekitar 1,5 juta baris dan periode 2019–2020 pada `data/README.md` belum dibuktikan oleh file ini. Lisensi sumber perlu dikonfirmasi sendiri.

Ambang kecepatan, asumsi kunci unik, satuan variabel, dan kotak pemeriksaan spasial harus ditinjau dengan metadata. Missing metadata dipertahankan; observasi pada konflik kunci dipilih berdasarkan kelengkapan, bukan penetapan kebenaran. Untuk Milestone 3, periksa jumlah sampel valid sebelum menghitung rerata kecepatan atau pola spasial.
"""
display(Markdown(summary_text))
(REPORT_DIR / "catatan_milestone2.md").write_text(summary_text, encoding="utf-8")
connection.close()

# Hasil aktual
- Data awal 775,458 baris, ukuran 72.98 MB, dengan 11 kolom sumber.
- Missing terbesar setelah normalisasi: location, sebanyak 540,981 (69.763%).
- Kecepatan di luar asumsi 0–120: 121 baris sebelum deduplikasi; nilai dijadikan null dan nilai asal disimpan.
- Karantina: 0 baris. Duplikat yang disisihkan: 48,384 baris.
- Hasil akhir: 727,074 baris; 149 kelompok memiliki konflik payload pada kunci yang sama.
- Periode aktual hasil bersih: 2019-11-26 14:00:00 sampai 2019-11-26 17:59:59.
- Semua aturan validasi dan uji baca ulang Parquet lulus.   

# Keterbatasan dan tindak lanjut
Data yang disertakan belum memenuhi syarat >=500 MB atau >1 juta baris. Lengkapi data asli sebelum pengumpulan jika belum memenuhi; jangan menduplikasi baris untuk mengejar syarat. Klaim sekitar 1,5 juta baris dan periode 2019–2020 pada data/README.md belum dibuktikan oleh file ini. Lisensi sumber perlu dikonfirmasi sendiri.

Ambang kecepatan, asumsi kunci unik, satuan variabel, dan kotak pemeriksaan spasial harus ditinjau dengan metadata. Missing metadata dipertahankan; observasi pada konflik kunci dipilih berdasarkan kelengkapan, bukan penetapan kebenaran. Untuk Milestone 3, periksa jumlah sampel valid sebelum menghitung rerata kecepatan atau pola spasial.